# Session 06 Topic 05: Reading intervals honestly

Use this notebook while working through Topic 05.

You can now build intervals and compare them. This notebook is about the judgement
required to report them responsibly — the part that separates an analyst from a
calculator.

The data changes here: instead of train patronage we use the student results extract
from Sessions 4 and 5. The questions get harder, and several of them have no single
right answer.

The two functions you wrote in Topic 04 are supplied, so you can get straight to the
analysis.

## 1. Setup

The extract needs one cleaning step before use. It contains repeated rows for the same
student and unit, so we remove exact duplicates on the columns that identify an
attempt, then drop rows with no mark.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="colorblind")

DATA_FOLDER = Path("data")
MARKS_FILE = DATA_FOLDER / "Y1_T1_2025.xlsx"

marks = pd.read_excel(MARKS_FILE, sheet_name="Extract")

# One row per student per unit attempt.
marks = marks.drop_duplicates(
    subset=["student_id", "unit_code", "attempt_no", "availability_year", "study_period"]
)

# Withdrawn students have no mark, so they cannot contribute to an average.
marks = marks[marks["mark"].notna()]

print("Marked unit attempts:", len(marks))
print("Students:", marks["student_id"].nunique())
print("Units:", marks["unit_code"].nunique())

In [ ]:
def interval_table(data, group_column, groups, value_column="mark"):
    """Return the mean and 95% interval for each group, as a table.

    The same function from Topic 04, with the value column now configurable.
    """
    rows = []

    for group in groups:
        values = data.loc[data[group_column] == group, value_column]

        n = len(values)
        mean = values.mean()
        standard_error = values.std() / np.sqrt(n)

        low, high = stats.t.interval(0.95, df=n - 1, loc=mean, scale=standard_error)

        rows.append({
            "group": group,
            "n": n,
            "mean": round(mean, 2),
            "low": round(low, 2),
            "high": round(high, 2),
        })

    return pd.DataFrame(rows)


def plot_intervals(summary, title, xlabel, highlight=None):
    """Draw one horizontal interval per group."""
    if highlight is None:
        highlight = []

    plt.figure(figsize=(8.5, 0.75 * len(summary) + 1.8))

    positions = range(len(summary) - 1, -1, -1)

    for position, row in zip(positions, summary.itertuples()):
        colour = "#D55E00" if row.group in highlight else "#0072B2"

        plt.plot([row.low, row.high], [position, position], color=colour, linewidth=2.5)
        plt.plot([row.low, row.low], [position - 0.12, position + 0.12], color=colour, linewidth=2.5)
        plt.plot([row.high, row.high], [position - 0.12, position + 0.12], color=colour, linewidth=2.5)
        plt.plot(row.mean, position, "o", color=colour, markersize=8)

    labels = summary["group"] + " (n=" + summary["n"].astype(str) + ")"
    plt.yticks(list(positions), labels)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.show()


print("Functions ready.")

## 2. Can the campuses be ranked reliably by mean mark?

A course director asks the question. Build the intervals and look.

In [ ]:
# Get the list of campus names from the location_name column.
# Call interval_table on marks, grouping by location_name.
# Sort the result by mean, highest first, and reset the index.

In [ ]:
plot_intervals(
    campus_summary,
    "Mean mark by campus, Term 1 2025",
    "Mean mark (95% confidence interval)",
)

The campuses form **two clear tiers**. Hanoi and Online sit above the other three, and
nothing overlaps across that divide. But campuses within each tier cannot be reliably
separated. The evidence supports two groups, not a ranking of all five campuses.

In [ ]:
# Write a small function compare(first_name, second_name) that:
#   - pulls out the marks for each campus
#   - prints the gap between the means and the p-value from stats.ttest_ind
# Call it on Hanoi vs Online, Melbourne North vs Perth Bentley,
#   and Hanoi vs Melbourne Park.

### Activity — Rewrite the ranking


A draft report lists the five campuses in order and calls Melbourne North "the
lowest-performing campus". Rewrite that sentence to reflect the intervals.

**Your answer:** Double-click this cell and replace this text with your response.

## 3. Small groups say very little

Now split the same data by a field where one category is very small.

In [ ]:
# Get the list of values in the gender column.
# Build the interval table for them.
# Add a width column: high minus low, rounded to 2 decimals.

In [ ]:
plot_intervals(
    gender_summary,
    "A small group gives a wide interval and no conclusion",
    "Mean mark (95% confidence interval)",
    highlight=["Gender X"],
)

The third group has 66 records. Its interval spans nearly six marks — more than ten
times the width of the largest group's — and it overlaps both others completely.

It would continue to overlap even if that group's true mean were considerably higher or
lower than everyone else's. The only honest conclusion is that **the data cannot say**.

This matters beyond statistics. Publishing a mean of 59.85 for that group alongside the
others, without its interval, invites comparisons the data cannot support — about a
group small enough that individuals might be identifiable.

In [ ]:
# How bad does this get on a field with many small categories?
country_counts = marks["country_of_birth"].value_counts()

print("Countries in the data:", len(country_counts))
print("With more than 500 students:", (country_counts > 500).sum())
print("With fewer than 30 students:", (country_counts < 30).sum())
print()
print(country_counts.to_string())

### Activity — Decide what to publish


A dashboard shows mean mark by country of birth. Look at the counts above and describe
how you would present this, and why.

**Your answer:** Double-click this cell and replace this text with your response.

## 4. Detectable is not the same as meaningful

Now the opposite problem. Compare Female and Male: the intervals barely overlap.

In [ ]:
# Pull out the marks for Female and for Male.
# Work out the gap between the means.
# Run stats.ttest_ind with equal_var=False and get the p-value.
# Print both means with their group sizes, the gap, and the p-value.

By the rule from Topic 04, p = 0.006 is evidence of a difference. The difference is
**0.87 marks**.

Nobody would change a policy over 0.87 marks. It is smaller than the effect of
rounding, smaller than the variation between two markers, and far smaller than the
spread within either group — which the next cell makes obvious.

In [ ]:
print("Gap between the group means:      ", round(gap, 2), "marks")
print("Standard deviation within Female: ", round(female.std(), 2), "marks")
print("Standard deviation within Male:   ", round(male.std(), 2), "marks")

The difference between the groups is about one fifteenth of the variation *inside* each
group. Knowing a student's gender tells you essentially nothing about their mark.

This is what very large samples do. With n in the thousands the standard error becomes
tiny, and even negligible differences become detectable. **A small p-value says a gap
this large is difficult to explain by random sampling variation if the population means
are equal. It does not say whether the difference is big enough to matter.**

So always report two things — the **size** of the difference in the units the client
cares about, and the **uncertainty** around it — and let the reader judge importance.

### Activity — Size against significance


For each pair, state whether the difference is detectable, whether it is meaningful, and
what you would tell the client.

1. Monday against Thursday on Sandringham: about 8,950 boardings, p < 0.0001.
2. Female against Male marks: 0.87 marks, p = 0.006.
3. Tuesday against Wednesday on Sandringham: 735 boardings, p = 0.42.

**Your answer:** Double-click this cell and replace this text with your response.

## 5. What population are you really describing?

Topic 01 raised this and set it aside. It needs settling before you report anything.

Both datasets contain every record. The intervals are meaningful only within the framing
that each observation is a single outcome of a repeatable process. That framing carries
conditions worth stating in a report:

- **The process must be stable.** A 2023-24 train interval says nothing useful about
  2019, because the pandemic permanently changed travel behaviour. Pooling all six years
  would yield an interval that describes no real period at all.
- **Observations should not be too dependent.** Consecutive days resemble each other —
  a strike or a heatwave affects a run of them — and standard errors assume
  independence.
- **The group must be the one you mean.** This extract covers three first-year units in
  a single term at a single institution. It supports claims about that cohort. It does not
  support claims about "engineering students" generally.

> **When an interval is the wrong tool.** If your question is purely about the records
> in hand — "what mark did student 8522991 get?" — you have the answer exactly, and no
> interval is needed. Confidence intervals are for claims that reach beyond the data
> you hold.

## 6. Ask the question first

There are ten possible pairwise comparisons among five campuses. Run all ten looking
for the smallest p-value, and you will often find one below 0.05, **even when there is
nothing there at all**.

The cell below proves it. It takes one campus — a single group of students with no real
subdivisions — splits them at random into five fake groups, and runs all ten
comparisons. Any "difference" it finds is pure noise, because the groups were made up.

Repeating that 300 times takes a few seconds to run.

In [ ]:
from itertools import combinations

melbourne_park = marks.loc[marks["location_name"] == "Melbourne Park", "mark"].to_numpy()

rng = np.random.default_rng(1106)

trials = 300
found_something = 0

for trial in range(trials):
    # Assign every student to one of five groups at random.
    fake_group = rng.integers(0, 5, size=len(melbourne_park))

    smallest_p = 1.0
    for first, second in combinations(range(5), 2):
        p = stats.ttest_ind(
            melbourne_park[fake_group == first],
            melbourne_park[fake_group == second],
            equal_var=False,
        ).pvalue
        smallest_p = min(smallest_p, p)

    if smallest_p < 0.05:
        found_something = found_something + 1

print("Trials where at least one pair looked 'significant':", found_something, "out of", trials)
print("That is", round(found_something / trials * 100, 1), "percent")

**More than a quarter of the time** — 82 of 300 trials — searching ten comparisons in
data with no real differences turns up at least one that passes p < 0.05.

That is not a flaw in the t-test. It is what happens when you *search*. Each test has
its own small chance of a false alarm, and running ten gives ten chances.

But — and this is the important part — **the problem is not how many comparisons exist.
It is why you made the one you report.**

- **A stated question.** The planner in Topic 04 asked about Mondays *before* opening
  the data, because working patterns changed after 2020. One comparison, chosen for a
  reason, and its p-value means what it says.
- **Fishing.** Computing every pair, noticing that Tuesday and Thursday came out
  furthest apart, and reporting that as a finding is a different activity. The
  comparison was selected *because* it looked impressive, so its p-value no longer means
  what it appears to.

The same arithmetic on the same data supports a conclusion in the first case but not
in the second. What changed is the order of operations.

This is the discipline Session 5 introduced and applied to comparisons: there, you
stated the question and predicted the relationship's shape before fitting a model.
Here, you state which groups you are comparing and why before you calculate.

Exploring freely is entirely legitimate — that is what Session 4's EDA was for. The
rule is about what you then **claim**. A pattern you noticed while exploring is a
question worth asking, not a finding. Report it as something to check against fresh
data, and say plainly that it came from exploration.

### Activity — Stated or fished?


For each, decide whether the conclusion is trustworthy, and say why.

1. A planner asks whether Mondays are quieter, tests Monday against the weekday
   average, and reports p < 0.0001.
2. An analyst compares all six financial years against each other, finds one pair with
   p = 0.04, and reports a change in that year.
3. An analyst notices while exploring that Alamein Fridays look low, then checks Fridays
   on the other two lines.

**Your answer:** Double-click this cell and replace this text with your response.

## 7. The caveats that travel with these numbers

Any report using the train data should carry these:

- Passenger counts are **rounded to the nearest 10** in the source, which limits how
  precisely small differences can be read.
- A few raw counts are **negative** — publisher corrections, not errors.
- The **2020 to 2022 collapse is real**, so any interval spanning those years describes
  a mixture of two different worlds.
- Counts are **boardings, not passengers**. One person changing trains boards twice.
- The three lines are a **deliberate selection** from 22.

And for the marks data:

- One term, one institution, three first-year units.
- Withdrawn students have no mark and are excluded, so the averages describe students
  who completed.

### Activity — Write the limitations


Write a short limitations paragraph for a report on the Sandringham weekday finding,
covering what the data can and cannot support.

**Your answer:** Double-click this cell and replace this text with your response.

## What you have done

- Found that five campuses form two groups, not a ranking of five
- Seen a group of 66 produce an interval too wide to support any conclusion
- Met a difference that is real and simultaneously too small to matter
- Watched a search of ten comparisons find "significance" in pure noise more than a
  quarter of the time
- Established that the problem is fishing, not counting — a pre-stated question is
  trustworthy, the same comparison chosen after the fact is not
- Written a limitations paragraph

That completes the Python work for Session 6, and for the unit. From Session 7 the work
moves to Power BI — where the charts build themselves, and the judgement you have
practised here matters more, not less.